# X-learner, DR-learner, DML

Qini curves for the three cross-fit / doubly robust estimators.

In [ ]:
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

from upliftbench.config import (
    CRITEO_PARQUET,
    FEATURES,
    MODELS_DIR,
    OUTCOME_VISIT,
    SEED,
    TEST_FRAC,
    TREATMENT_COL,
)
from upliftbench.data.loader import train_test_split_rct
from upliftbench.eval.qini import qini_coefficient, qini_curve
from upliftbench.persistence import load_model

df = pq.read_table(
    str(CRITEO_PARQUET), columns=[*FEATURES, TREATMENT_COL, OUTCOME_VISIT]
).to_pandas()
_, test_idx = train_test_split_rct(n=len(df), test_frac=TEST_FRAC, seed=SEED)
X = df[FEATURES].to_numpy()[test_idx]
T = df[TREATMENT_COL].to_numpy()[test_idx]
Y = df[OUTCOME_VISIT].to_numpy()[test_idx]
fig, ax = plt.subplots(figsize=(7, 5))
for name in ("x-learner", "dr-learner", "dml"):
    mpath = sorted(MODELS_DIR.glob(f"{name}_*.joblib"))[-1]
    est = load_model(mpath)
    cate = est.predict_cate(X)
    xs, ys = qini_curve(T, Y, cate)
    ax.plot(xs, ys, label=f"{name} (Qini {qini_coefficient(T, Y, cate):+.3f})")
ax.set_xlabel("Population fraction")
ax.set_ylabel("Cumulative uplift")
ax.legend()
ax.set_title("Cross-fit estimators")
fig